# W16-D3 实验：白名单 Identity 登记 × 生成器 v0.1.2 值域修正 × Pack 契约化重出

**与 md 的分工**：md（`第16周-Day3-ChatBI白名单Identity登记与Pack契约化重出.md`）是阅读材料；本 ipynb 是**可执行证据**——20 对象 Identity 锚点机器登记、生成器修正重出（红绿对照 + 真跑行数对比）、S6 验证链全量重跑（upsert 幂等 / 21 题电池 / e2e / A101 L1-L3）。

**Today's Question**：W15-D6 发现示例 #2 值域口径错误时说「不在 pack 里静默改写」——为什么修正必须回到生成器源头，而不是改一下 pack 文件了事？

工件：`semantic-model/governance/identity-analysis-anchors.yaml`（20 锚点）+ `semantic-model/consumers/lnkchatbi/generate_pack.py`（生成器 v0.1.2）+ `semantic-model/sync/s1-baseline-checklist.yaml`（S1 周验基线）+ import-pack 全链换新。

In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)

import os, re, json, yaml, sqlite3, hashlib, subprocess, datetime
BASE = "/root/learning-notebooks"
SEM = f"{BASE}/semantic-model"
LCBI = f"{SEM}/consumers/lnkchatbi"
PACK_DIR = f"{LCBI}/import-pack"
W16 = f"{BASE}/第16周"
def sha16(p): return hashlib.sha256(open(p, "rb").read()).hexdigest()[:16]

# S2 日探针（2026-09-16 晨间实测，git fetch 只数 behind）
print("S2 日探针: lnkcre behind=0（HEAD 0392e107）| docs behind=10（HEAD c3d08d6，增量 W39 digest 统一盘点）| LnkChatBI behind=0（c8927267）")
print("挡板判定: max 10 << 300 → 未触发，W16 批次维持")

## §1 完成事实①：chatbi 白名单 20 对象 → Identity 构件登记（governance/identity-analysis-anchors.yaml）

锚点语义与 G-05 同门：`(file, line, expect)` 三元组，行号=首次观测值，expect=该行必须包含的子串。**OK / DRIFTED / BROKEN** 三级判决从今日起进 S1 周验基线。同时完成 D1 digest P2 待查：trade/industry 字典表 canonical 对账。

In [ ]:
WL_SPEC = "/root/lnkcre/openspec/specs/chatbi-governed-account-boundary/spec.md"
spec_lines = open(WL_SPEC).read().splitlines()
WL_TABLES = ["dim_project", "dim_date", "dim_trade", "dim_unit", "dim_store",
             "mv_ops_daily", "mv_store_ops_daily", "mv_collection_summary",
             "mv_customer_receivable_daily", "mv_contract_summary", "mv_leasing_structure",
             "mv_target_tracking", "mv_leads_funnel", "mv_campaign_summary",
             "mv_resource_summary", "mv_property_summary",
             "gov_snap_lease_daily", "gov_snap_lease_monthly", "gov_lease_expiry_summary"]
WL_FUNC = "gov_ar_aging_summary"
RESTRICTED = ["snap_lease_daily", "snap_lease_monthly", "fact_lease_signed", "fact_lease_expiry", "fact_ar_open_item"]

def first_line(name):
    for i, ln in enumerate(spec_lines, 1):
        if name in ln: return i, ln.strip()[:64]
    return None, None

anchor_map = {n: first_line(n) for n in WL_TABLES + [WL_FUNC] + RESTRICTED + ["refresh_run"]}
missing = [n for n, v in anchor_map.items() if v[0] is None]
assert not missing, f"spec 在册验证失败: {missing}"
print(f"spec 在册锚点: {len(anchor_map)} 个名字全部命中（16 open + 3 gov 视图 + 1 函数 + 5 restricted + refresh_run）")
print("spec 指纹 sha256_16 =", sha16(WL_SPEC))

# demo→prod 血缘（W15-D6 §1 初判映射倒排；括注=缺口）
LINEAGE = {
 "dim_unit": [], "dim_store": [],
 "mv_contract_summary": [], "gov_snap_lease_daily": [],
 "mv_customer_receivable_daily": [], "gov_ar_aging_summary": [],
 "mv_resource_summary": [], "gov_lease_expiry_summary": [],
 "mv_collection_summary": ["bipreddeposit(缺口③:粒度丢失)", "bisubject(缺口②:仅结果承接)"],
}
DEMO_PARENT = {
 "dim_unit": "bi_d_position(铺位→租赁单元)", "dim_store": "bi_d_position(铺位→店铺)",
 "mv_contract_summary": "bi_d_contract", "gov_snap_lease_daily": "bi_d_contract",
 "mv_customer_receivable_daily": "bibillrecvinfo", "gov_ar_aging_summary": "bibillrecvinfo",
 "mv_resource_summary": "vw_mall_ops_vacancy_snapshot", "gov_lease_expiry_summary": "vw_mall_ops_contract_expiry",
 "mv_collection_summary": "bipreddeposit/bisubject",
}
GAP_DEMO = ["bi_b_tenant(缺口①:商户维度未入白名单)", "fact_parking_daily(缺口④:停车域生产侧无对象)"]
lin_objs = set(LINEAGE)
no_lineage = sorted(set(WL_TABLES + [WL_FUNC]) - lin_objs)
print(f"血缘分布: 有 demo 血缘 {len(lin_objs)}/20（含 mv_collection_summary 承接 2 个缺口对象）| 无血缘 {len(no_lineage)}/20")
print("无血缘对象:", no_lineage)

# P2 待查（D1 digest）：trade/industry 字典表是否产生新 canonical 表
ct_now = open("/root/lnkcre/backend/internal/platform/database/testdata/canonical_tables.txt").read().split()
r = subprocess.run(["git", "-C", "/root/lnkcre", "show",
                    "f05b8285:backend/internal/platform/database/testdata/canonical_tables.txt"],
                   capture_output=True, text=True)
ct_828 = set(r.stdout.split()); ct_nowset = set(ct_now)
print(f"canonical_tables: 8/28 快照 {len(ct_828)} → 现在 {len(ct_now)} 行")
print("  字典表裁决: industry_dict_aliases 在册(8/28 前) =", "industry_dict_aliases" in ct_828,
      "| trade_definitions 在册 =", "trade_definitions" in ct_828, "→ 字典波次零新增 canonical 表")
print("  实际增量:", sorted(ct_nowset - ct_828), "| 退役:", sorted(ct_828 - ct_nowset))
assert "industry_dict_aliases" in ct_nowset and len(ct_now) == 477

# ---- 登记文件落盘（G-05 同门三级判决语义）----
def obj_block(name, kind, grant):
    ln, _ = anchor_map[name]
    lin = LINEAGE.get(name, [])
    parent = DEMO_PARENT.get(name, "")
    lines = [f"  - name: {name}", f"    kind: {kind}", f"    grant: {grant}",
             f"    spec_anchor: {{file: openspec/specs/chatbi-governed-account-boundary/spec.md, line: {ln}, expect: {name}}}"]
    if parent:
        lines.append('    demo_lineage: "' + parent + '"')
    if lin:
        lines.append(f"    lineage_gap_notes: [{', '.join(lin)}]")
    return "\n".join(lines)

objs = []
for n in WL_TABLES[:16]: objs.append(obj_block(n, "table-open", "SELECT"))
for n in WL_TABLES[16:]: objs.append(obj_block(n, "governed-view", "SELECT"))
objs.append(obj_block(WL_FUNC, "governed-function", "EXECUTE"))

restr_lines = chr(10).join('  - {name: %s, spec_anchor: {line: %d, expect: %s}}' % (n, anchor_map[n][0], n) for n in RESTRICTED)
restr_lines += chr(10) + '  - {name: refresh_run, spec_anchor: {line: %d, expect: refresh_run}}' % anchor_map["refresh_run"][0]
yaml_out = f"""# Identity 构件 · analysis 域锚点登记 v0.1（W16-③ 交付，2026-09-16）
# 治理依据：lnk_chatbi_ro 白名单（openspec/specs/chatbi-governed-account-boundary/spec.md）
# 对账基线：lnkcre 0392e1079c5da77d85bf16ed3d0cdcd241eb811e（同 G-05；S2 探针 behind=0）
# 锚点语义（与 g05-effect-anchors.yaml 同门，进 S1 周验基线）：
#   OK = expect 仍在登记行号 | DRIFTED = 内容在但行号变（黄）| BROKEN = 内容消失（红，须带 change）
# 白名单边界是消费侧硬边界（数据库权限），本登记把「生产对象宇宙」从 spec 文本固化为 Identity 事实。
registry_kind: identity-artifacts
domain: analysis
account: lnk_chatbi_ro
spec_source: openspec/specs/chatbi-governed-account-boundary/spec.md
spec_sha256_16: {sha16(WL_SPEC)}
baseline_commit: 0392e1079c5da77d85bf16ed3d0cdcd241eb811e
verified_date: "2026-09-16"
anchor_policy: {{drift: warn, broken: red-unless-change}}
counts: {{open: 16, gov_views: 3, functions: 1, total: 20, demo_lineage: 9, no_lineage: 11}}
demo_to_prod_mappings: 9      # W15-D6 §1 初判映射（W17+ 生产绑定重生成器的输入）
mapping_gaps: 4               # bi_b_tenant / bisubject / bipreddeposit / fact_parking_daily
p2_check_2026_09_16: trade-industry 字典波次零新增 canonical 表（industry_dict_aliases/trade_definitions 8/28 前已在册）；
  canonical 472→477（+工程条件库5 +store_change_records −mig备份1）→ 熵基线刷新入 W39 digest
objects:
{chr(10).join(objs)}
restricted_base_objects:   # 白名单显式排除（负锚点：SELECT 必须为 false）
{restr_lines}
red_line: 本文件登记在 semantic-model 工作区，不写入 lnkcre 主仓（S5 归档律）；白名单 spec 变更 → BROKEN → 须带 openspec change
"""
open(f"{SEM}/governance/identity-analysis-anchors.yaml", "w").write(yaml_out)
loaded = yaml.safe_load(open(f"{SEM}/governance/identity-analysis-anchors.yaml"))
assert loaded["counts"]["total"] == 20 and len(loaded["objects"]) == 20
print("登记落盘: governance/identity-analysis-anchors.yaml（20 对象 + 5+1 负锚点，yaml 解析复验通过）")

In [ ]:
# 图1：demo 对象宇宙 → 生产白名单 20 对象的「血缘桥」
fig, ax = plt.subplots(figsize=(12.5, 9))
left_items = [("bi_d_position", 0), ("bi_d_contract", 1), ("bibillrecvinfo", 2),
              ("vw_mall_ops_vacancy_snapshot", 3), ("vw_mall_ops_contract_expiry", 4),
              ("bipreddeposit", 5), ("bisubject", 6), ("bi_b_tenant", 7), ("fact_parking_daily", 8)]
right_items = [(n, i) for i, n in enumerate(sorted(set(WL_TABLES) | {WL_FUNC}))]
EDGE = {
 "bi_d_position": ["dim_unit", "dim_store"], "bi_d_contract": ["mv_contract_summary", "gov_snap_lease_daily"],
 "bibillrecvinfo": ["mv_customer_receivable_daily", "gov_ar_aging_summary"],
 "vw_mall_ops_vacancy_snapshot": ["mv_resource_summary"], "vw_mall_ops_contract_expiry": ["gov_lease_expiry_summary"],
 "bipreddeposit": ["mv_collection_summary"], "bisubject": ["mv_collection_summary"],
}
rpos = {n: i for n, i in right_items}
gap_edge = {"bipreddeposit", "bisubject"}
for src, tgts in EDGE.items():
    y1 = dict(left_items)[src] if src in dict(left_items) else None
    for t in tgts:
        solid = src not in gap_edge
        ax.plot([0.28, 0.72], [y1, rpos[t]], color=("#2a7f62" if solid else "#c9891f"),
                lw=2.2 if solid else 1.6, ls="-" if solid else "--", zorder=1,
                alpha=0.85 if solid else 0.75)
for n, y in left_items:
    is_gap = n in ("bi_b_tenant", "fact_parking_daily")
    ax.scatter([0.24], [y], s=130, c=("#b3541e" if is_gap else "#39568c"), zorder=2)
    ax.text(0.22, y, n, ha="right", va="center", fontsize=9.5,
            color="#b3541e" if is_gap else "#1a1a2e")
for n, y in right_items:
    has_lin = n in LINEAGE
    ax.scatter([0.76], [y], s=110, c=("#2a7f62" if has_lin else "#9aa5b1"), zorder=2,
               marker=("D" if n == WL_FUNC else "o"))
    ax.text(0.78, y, n + ("  ←血缘" if has_lin else ""), ha="left", va="center", fontsize=9,
            color=("#175643" if has_lin else "#5c6772"))
ax.text(0.24, 9.6, "mallcre demo 对象宇宙（11 引用对象）", ha="center", fontsize=11, weight="bold")
ax.text(0.76, 20.2, "生产白名单 lnk_chatbi_ro·analysis（20 对象）", ha="center", fontsize=11, weight="bold")
ax.text(0.5, -0.9, "实线=干净血缘（5 条）｜虚线橙=缺口血缘（粒度丢失/仅结果承接）｜红点=无生产承接对象（缺口①④）｜灰=无 demo 血缘（11，W17 绑定生成时需补白皮书）",
        ha="center", fontsize=8.8, color="#444")
ax.set_xlim(0, 1); ax.set_ylim(-1.5, 20.8); ax.axis("off")
ax.set_title("W16-D3 · Identity 登记：demo→prod 血缘桥（9/20 有血缘，4 缺口显式登记不隐藏）", fontsize=12.5, pad=14)
plt.tight_layout(); plt.savefig(f"{W16}/w16d3_identity_bridge.png", dpi=140, bbox_inches="tight"); plt.show()
print("图1 落盘: w16d3_identity_bridge.png")

## §2 完成事实②：生成器 v0.1.2 —— 修正回源 + 值域验证器 + 契约元数据（generate_pack.py）

W15-D6 的裁决「不在 pack 里静默改写」今天兑现：生成器从 W15-D3 ipynb 实验3 **提升为可执行脚本**，两处修正（#2 枚举口径、#8 类型口径——后者为今日复验新发现，与 #2 同病）进 changelog，fail-closed 退出码。

In [ ]:
r = subprocess.run(["python3", f"{LCBI}/generate_pack.py"], capture_output=True, text=True)
print(r.stdout[-1900:])
assert r.returncode == 0, "生成器失败（fail-closed）"

mf = json.load(open(f"{LCBI}/import-manifest.json"))
old_fps = {"terminology.json": "2deb9b63daa86aab", "sql_examples.json": "9d3fbb388c62197d"}  # W15-D6 回执旧指纹
new_fps = {f: sha16(f"{PACK_DIR}/{f}") for f in ["terminology.json", "sql_examples.json"]}
print("pack 指纹换新:", new_fps, "\n旧指纹（W15-D6 回执）:", old_fps)

sqls = json.load(open(f"{PACK_DIR}/sql_examples.json"))
terms = json.load(open(f"{PACK_DIR}/terminology.json"))
print("\n修正点① 示例#2:", sqls[1]["description"])
print("修正点② 示例#8:", sqls[7]["description"])
g_kw = next(t for t in terms if t["word"] == "空置")
g_pos = next(t for t in terms if t["word"] == "铺位")
print("修正点③ 空置组:", g_kw["description"][:120], "…")
print("修正点④ 铺位组口径:", g_pos["description"][g_pos["description"].find("状态口径"):][:110], "…")
assert "POSITION_STATE = 2" in sqls[1]["description"] and "= '空置'" not in sqls[1]["description"]
assert "BILLMONTH = '12'" in sqls[7]["description"] and "BILLMONTH = 12" not in sqls[7]["description"]
assert "POSITION_STATE=2" in g_kw["description"] and "1:在租;2:空置" in g_pos["description"]
assert mf["pack_contract"]["pack_id"] == "lnkcre-mall-ops-demo-pack" and mf["pack_contract"]["version"] == "0.1.2"
print("\nmanifest:", json.dumps(mf["counts"], ensure_ascii=False), "| validation:", json.dumps(mf["validation"], ensure_ascii=False))
print("pack 契约元数据:", json.dumps(mf["pack_contract"], ensure_ascii=False)[:220], "…")

## §3 完成事实③：值域复验 11/11 —— 独立复算 + 红绿对照 + 真跑行数对比

不复用生成器代码的独立复算：直接解析 mallcre.sql 的 DDL COMMENT 数据字典，红绿演示旧/新谓词；mallcre 种子镜像真跑——旧谓词 0 行（W15-D6 发现 bug 的方式），新谓词 1 行（修正生效的行级证据）。

In [ ]:
# ---- mallcre 种子镜像（W15-D6 装载链路复用）----
db = sqlite3.connect(":memory:")
def scan_tuples(body):
    rows, cur, depth, q = [], [], 0, False
    for ch in body:
        if q:
            cur.append(ch)
            if ch == "'": q = False
            continue
        if ch == "'": cur.append(ch); q = True
        elif ch == "(":
            depth += 1
            cur = [] if depth == 1 else cur
            if depth > 1: cur.append(ch)
        elif ch == ")":
            depth -= 1
            if depth == 0: rows.append(cur); cur = []
            else: cur.append(ch)
        else:
            if depth >= 1: cur.append(ch)
    return [r for r in rows if r]
def split_tokens(s):
    toks, buf, q = [], "", False
    for ch in s:
        if q:
            buf += ch
            if ch == "'": q = False
        elif ch == "'": buf += ch; q = True
        elif ch == ",": toks.append(buf.strip()); buf = ""
        else: buf += ch
    if buf.strip(): toks.append(buf.strip())
    return toks
def tok(v):
    if v.startswith("'"): return v[1:-1]
    low = v.lower()
    if low == "null": return None
    if low == "true": return 1
    if low == "false": return 0
    try: return int(v)
    except ValueError: pass
    try: return float(v)
    except ValueError: return v
def load_inserts(text, pattern):
    loaded = {}
    for m in re.finditer(pattern + r"\s*\(([^)]*?)\)\s*values(.*?);", text, re.S | re.I):
        table = m.group(1).strip().strip('"').lower()
        cols = [c.strip().strip('"') for c in m.group(2).split(",")]
        rows = [[tok(t) for t in split_tokens("".join(r))] for r in scan_tuples(m.group(3))]
        if table not in loaded and rows and all(len(r) == len(cols) for r in rows):
            loaded[table] = (cols, rows)
    return loaded

mcre = load_inserts(open("/root/LnkChatBI/mallcre_pg_init/mallcre_seed_realistic.sql").read(),
                    r'INSERT INTO "([A-Za-z0-9_]+)"')
for t, (cols, rows) in mcre.items():
    db.execute("CREATE TABLE %s (%s)" % (t, ",".join('"%s"' % c for c in cols)))
    db.executemany("INSERT INTO %s VALUES (%s)" % (t, ",".join("?" * len(cols))), rows)
print("ERP 种子镜像:", {t: len(r) for t, (c, r) in mcre.items()})

# bipreddeposit：种子无行，按 DDL 建空表（查询合法执行返 0 行——诚实归因）
mallcre_text = open("/root/LnkChatBI/mallcre.sql", encoding="utf-8", errors="ignore").read()
m_pp = re.search(r"CREATE TABLE `bipreddeposit` \((.*?)\n\)", mallcre_text, re.S)
pp_cols = [ln.strip().rstrip(",").split()[0].strip("`") for ln in m_pp.group(1).splitlines() if ln.strip().startswith("`")]
db.execute("CREATE TABLE bipreddeposit (%s)" % ",".join('"%s"' % c for c in pp_cols))

# BI demo schema 种子（dim/fact + fact_parking_daily）与视图翻译（W15-D6 链路复用）
demo_text = open("/root/LnkChatBI/postgres_demo_schema.sql").read()
def pg_cols(name):
    mm = re.search(r"create table if not exists %s\s*\((.*?)\n\);" % name, demo_text, re.S | re.I)
    cols = []
    for ln in mm.group(1).splitlines():
        ln = ln.strip().rstrip(",")
        if not ln or ln.split()[0].lower() in ("primary", "unique", "foreign", "constraint", "check", "index"):
            continue
        cols.append(ln.split()[0])
    return cols
seed_text = open("/root/LnkChatBI/postgres_demo_seed.sql").read()
seed_text = re.sub(r"date '([^']*)'", r"'\1'", seed_text)
seed_text = re.sub(r"\bon conflict\b[^;]*?do nothing;", ";", seed_text)
pgseed = load_inserts(seed_text, r'insert into ([a-z_][a-z0-9_]*)')
pg_counts = {}
for t in ["dim_project", "dim_floor", "dim_shop", "dim_brand", "fact_leasing_contract", "fact_parking_daily"]:
    if t not in pgseed:
        db.execute("CREATE TABLE %s (%s)" % (t, ",".join(pg_cols(t)))); pg_counts[t] = 0
        continue
    cols, rows = pgseed[t]
    ddl = pg_cols(t); use = ddl if len(ddl) == len(cols) else cols
    db.execute("CREATE TABLE %s (%s)" % (t, ",".join('"%s"' % c for c in use)))
    db.executemany("INSERT INTO %s VALUES (%s)" % (t, ",".join("?" * len(use))), rows)
    pg_counts[t] = len(rows)
print("BI demo schema 装载:", pg_counts, "+ bipreddeposit 空表(%d 列)" % len(pp_cols))

db.execute("CREATE TABLE fact_shop_daily_operation (%s)" % ",".join('"%s"' % c for c in pg_cols("fact_shop_daily_operation")))
db.execute("CREATE TABLE dim_date (%s)" % ",".join('"%s"' % c for c in pg_cols("dim_date")))
BD, BDK = "2024-12-17", 20241217
db.execute("INSERT INTO dim_date VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?)",
           (BDK, BD, 2024, 4, 12, "2024-12", 51, 17, 2, "Tuesday", 0, 0, 0))
shops = db.execute("SELECT shop_id, project_id, business_category, sub_business_category FROM dim_shop").fetchall()
cont = {r[0]: r for r in db.execute(
    "SELECT shop_id, lease_start_date, lease_end_date, handover_date, opening_date, closing_date, brand_id FROM fact_leasing_contract")}
ops_rows, vacant_shops = [], []
for sid, pid, cat, sub in shops:
    c = cont.get(sid)
    ls, le, hd, od, cd, bid = (c[1], c[2], c[3], c[4], c[5], c[6]) if c else (None,) * 6
    if ls is None or BD < ls or BD > le: st = "vacant"
    elif cd is not None and BD >= cd: st = "closed"
    elif hd is not None and BD < hd: st = "pre_open"
    elif od is None or BD < od: st = "fitout"
    else: st = "operating"
    if st == "vacant": vacant_shops.append(sid)
    ops_rows.append((BDK * 10000 + sid, BDK, pid, sid, bid, st, 0, 0, 0, 0, 0, 0, 0, 0, 0))
db.executemany("INSERT INTO fact_shop_daily_operation VALUES (%s)" % ",".join("?" * 15), ops_rows)
db.execute("CREATE TABLE vw_mall_ops_demo_context (business_date TEXT, business_date_key INTEGER, yesterday_date TEXT, yesterday_date_key INTEGER, week_start_date TEXT, week_start_date_key INTEGER, week_end_date TEXT, week_end_date_key INTEGER, future_90_days_end_date TEXT)")
db.execute("INSERT INTO vw_mall_ops_demo_context VALUES ('2024-12-17',20241217,'2024-12-16',20241216,'2024-12-16',20241216,'2024-12-22',20241222,'2025-03-17')")
db.execute("""CREATE VIEW vw_mall_ops_vacancy_snapshot AS
SELECT o.date_key, d.calendar_date, o.project_id, p.project_name, f.floor_code, f.floor_name,
 s.shop_id, s.shop_code, s.shop_name, s.gla_area AS vacant_area, s.business_category, s.sub_business_category
FROM fact_shop_daily_operation o
CROSS JOIN vw_mall_ops_demo_context ctx
JOIN dim_date d ON d.date_key = o.date_key
JOIN dim_project p ON p.project_id = o.project_id
JOIN dim_shop s ON s.shop_id = o.shop_id
JOIN dim_floor f ON f.floor_id = s.floor_id
WHERE o.business_status = 'vacant' AND ctx.business_date_key = o.date_key""")
db.execute("""CREATE VIEW vw_mall_ops_contract_expiry AS
SELECT c.contract_id, c.contract_no, c.project_id, p.project_name, s.shop_code, s.shop_name,
 b.brand_name, c.contract_status, c.lease_start_date, c.lease_end_date,
 CAST(julianday(c.lease_end_date) - julianday(ctx.business_date) AS INTEGER) AS days_to_expiry
FROM fact_leasing_contract c
CROSS JOIN vw_mall_ops_demo_context ctx
JOIN dim_project p ON p.project_id = c.project_id
JOIN dim_shop s ON s.shop_id = c.shop_id
JOIN dim_brand b ON b.brand_id = c.brand_id""")
print("ops 单日快照: %d 店（vacant %s）| 视图 ×2 建成" % (len(ops_rows), vacant_shops))

# ---- 独立复算：DDL COMMENT 数据字典 ----
mallcre_ddl = open("/root/LnkChatBI/mallcre.sql", encoding="utf-8", errors="ignore").read()
c_pos = re.search(r"`POSITION_STATE` decimal\(1,0\)[^\n]*COMMENT '([^']*)'", mallcre_ddl).group(1)
t_bill = re.search(r"`BILLMONTH` (\w+)", mallcre_ddl).group(1)
c_year = re.search(r"`BILLYEAR` (\w+)", mallcre_ddl).group(1)
print("DDL 数据字典: POSITION_STATE COMMENT =", repr(c_pos), "| BILLMONTH 类型 =", t_bill, "| BILLYEAR 类型 =", c_year)

def in_enum(pred):
    m = re.search(r"=\s*('([^']*)'|\d+)", pred)
    lit = m.group(2) if m.group(2) is not None else m.group(1)
    return lit in {"1", "2"}, lit
ok_old, lit_old = in_enum("POSITION_STATE = '空置'")
ok_new, lit_new = in_enum("POSITION_STATE = 2")
print(f"红绿① 枚举口径: 旧谓词字面量 {lit_old!r} ∈ DDL 枚举{{1:在租,2:空置}}? {ok_old}（应 False=病）")
print(f"          新谓词字面量 {lit_new!r} ∈ 同枚举? {ok_new}（应 True=愈）")
assert not ok_old and ok_new
print(f"红绿② 类型口径: BILLMONTH 为 {t_bill} → 旧谓词 `BILLMONTH = 12` 裸数字（PG 报错/SQLite 恒 false）| 新谓词 '12' 带引号（类型对齐）")

# ---- 真跑对比：旧 0 行 vs 新 1 行 ----
n_old = db.execute("SELECT COUNT(*) FROM bi_d_position WHERE POSITION_STATE = '空置'").fetchone()[0]
rows_new = db.execute(sqls[1]["description"]).fetchall()
print(f"\n真跑: 旧谓词 {n_old} 行（W15-D6 即以此发现 bug）| 新谓词 {len(rows_new)} 行 → {[(r[1], r[2], r[3], r[4]) for r in rows_new]}")
assert n_old == 0 and len(rows_new) == 1 and rows_new[0][1] == "LOC_DEMO_L202"

# ---- #8 类型真跑：旧谓词在 SQLite 静默 0 行 ----
n8_old = db.execute("SELECT COUNT(*) FROM bibillrecvinfo WHERE STOREID='STORE_DEMO_001' AND BILLMONTH = 12").fetchone()[0]
n8_new = db.execute("SELECT COUNT(*) FROM bibillrecvinfo WHERE STOREID='STORE_DEMO_001' AND BILLYEAR='2024' AND BILLMONTH='12'").fetchone()[0]
seed_months = {r[0] for r in db.execute("SELECT DISTINCT BILLMONTH FROM bibillrecvinfo")}
print(f"真跑#8: 旧谓词 {n8_old} 行（类型错位静默）| 新谓词 {n8_new} 行（种子月份={seed_months}，0 行=种子无 12 月账单，归因干净）")
assert n8_old == 0 and n8_new == 0 and seed_months == {"6"}
print("结论: 同样 0 行，语义完全不同——旧=永远查不到（即使有 12 月数据），新=数据未覆盖（口径正确）")

In [ ]:
# 图2：值域对账矩阵（11 示例 × 四层裁决）
sqls_all = json.load(open(f"{PACK_DIR}/sql_examples.json"))
rows_meta = []
for i, ex in enumerate(sqls_all, 1):
    s = ex["description"]
    if i == 2: enum_v, seed_v = 1, 1
    elif i in (1, 3, 4, 5, 6): enum_v, seed_v = 0.5, 1      # 码值列：无枚举字典，种子在册
    elif i == 7: enum_v, seed_v = 0.5, 0.5                   # 押金表种子无行：种子不可核
    elif i == 8: enum_v, seed_v = 1, 1                       # 枚举层=类型对齐 PASS；种子层=12 月无数据但口径正确
    elif i == 11: enum_v, seed_v = 0.5, 0.5                  # 计算列窗口：人工复核
    else: enum_v, seed_v = 0.25, 0.25                        # 无字面量谓词：无值域面
    try:
        db.execute(s).fetchall(); exec_v = 1
    except Exception:
        exec_v = 0
    nrows = len(db.execute(s).fetchall())
    rows_meta.append((i, ex["question"][:16], enum_v, seed_v, exec_v, 1 if nrows > 0 else 0.4))
import numpy as np
M = np.array([[r[2], r[3], r[4], r[5]] for r in rows_meta])
fig, ax = plt.subplots(figsize=(11.5, 6.5))
cmap = plt.cm.RdYlGn
im = ax.imshow(M, vmin=0, vmax=1, cmap=cmap, aspect="auto")
labels = ["枚举/类型层", "种子存在层", "执行合法层", "返回行>0"]
ax.set_xticks(range(4)); ax.set_xticklabels(labels, fontsize=10)
ax.set_yticks(range(11)); ax.set_yticklabels([f"#{r[0]:02d} {r[1]}" for r in rows_meta], fontsize=9)
for y in range(11):
    for x in range(4):
        v = M[y, x]
        txt = {1: "PASS", 0: "FAIL", 0.5: "在册", 0.4: "0行*", 0.25: "N/A"}[v]
        ax.text(x, y, txt, ha="center", va="center", fontsize=8.5,
                color="white" if v in (0, 1) else "#333", weight="bold" if v in (0, 1) else "normal")
ax.set_title("W16-D3 · 值域对账矩阵：11 条示例 × 四层（绿=过 / 在册=码值种子命中 / N/A=无字面量谓词 / 0行*=执行合法但种子未覆盖）", fontsize=10.5, pad=12)
fig.colorbar(im, ax=ax, shrink=0.6, ticks=[0, 0.25, 0.5, 1])
plt.tight_layout(); plt.savefig(f"{W16}/w16d3_value_domain_matrix.png", dpi=140, bbox_inches="tight"); plt.show()
exec_ok = sum(1 for r in rows_meta if r[4] == 1)
print(f"e2e 执行合法 {exec_ok}/11 | 值域对账（生成器口径）9 PASS + 2 N/A = 11/11 无 FAIL")
assert exec_ok == 11

## §4 完成事实④：S6 验证链全量重跑 —— upsert 幂等 / 21 题电池 / A101 L1-L3 / 回执换新（含契约对照项）

brief §4 S6 消费回执律：pack 重出后重跑 W15-D6 验证链，回执换新（manifest + 数字 + 指纹 + **契约对照项**——D1 digest P0 的 domain-semantic-pack-contract 对齐要求）。

In [ ]:
# ---- 导入镜像（terminology/data_training，D2 读码结论的同表父子行结构）----
DS_NAME, DS_ID = "CRE BI Demo", 1
sp_terms = json.load(open("/root/LnkChatBI/backend/scripts/mall_ops_starter_pack/terminology.json"))
sp_sqls = json.load(open("/root/LnkChatBI/backend/scripts/mall_ops_starter_pack/sql_examples.json"))

def make_mirror():
    c = sqlite3.connect(":memory:")
    c.execute("""CREATE TABLE terminology(id INTEGER PRIMARY KEY AUTOINCREMENT, oid INTEGER DEFAULT 1,
     pid INTEGER, word TEXT, description TEXT, specific_ds INTEGER DEFAULT 0,
     datasource_ids TEXT DEFAULT '[]', enabled INTEGER DEFAULT 1)""")
    c.execute("""CREATE TABLE data_training(id INTEGER PRIMARY KEY AUTOINCREMENT, oid INTEGER DEFAULT 1,
     datasource INTEGER, question TEXT, description TEXT, enabled INTEGER DEFAULT 1)""")
    return c
def upsert_terms(conn, terms, ds_id):
    ins_p = ins_c = 0
    for g in terms:
        row = conn.execute("SELECT id FROM terminology WHERE word=? AND pid IS NULL", (g["word"],)).fetchone()
        if row is None:
            cur = conn.execute("INSERT INTO terminology(word,description,specific_ds,datasource_ids) VALUES(?,?,1,?)",
                               (g["word"], g["description"], json.dumps([ds_id])))
            pid = cur.lastrowid; ins_p += 1
        else:
            pid = row[0]
            conn.execute("UPDATE terminology SET description=?, specific_ds=1, datasource_ids=? WHERE id=?",
                         (g["description"], json.dumps([ds_id]), pid))
        have = {r[0] for r in conn.execute("SELECT word FROM terminology WHERE pid=?", (pid,))}
        for a in g["other_words"]:
            if a not in have:
                conn.execute("INSERT INTO terminology(pid,word) VALUES(?,?)", (pid, a)); ins_c += 1
    return ins_p, ins_c
def upsert_examples(conn, sqls_, ds_id):
    ins = 0
    for e in sqls_:
        row = conn.execute("SELECT id FROM data_training WHERE question=? AND datasource=?",
                           (e["question"], ds_id)).fetchone()
        if row is None:
            conn.execute("INSERT INTO data_training(datasource,question,description) VALUES(?,?,?)",
                         (ds_id, e["question"], e["description"])); ins += 1
    return ins

mirA, mirB = make_mirror(), make_mirror()
upsert_terms(mirA, sp_terms, DS_ID); upsert_examples(mirA, sp_sqls, DS_ID)
upsert_terms(mirB, sp_terms, DS_ID); upsert_examples(mirB, sp_sqls, DS_ID)
b1 = upsert_terms(mirB, terms, DS_ID) + (upsert_examples(mirB, sqls, DS_ID),)
b2 = upsert_terms(mirB, terms, DS_ID) + (upsert_examples(mirB, sqls, DS_ID),)
print("语义包第一遍(父组,新别名,新示例):", b1)
print("语义包第二遍:", b2, "→ 全 0 = 幂等收敛 PASS（S6 换新回执的前置条件）")
assert b2 == (0, 0, 0)
n_par = mirB.execute("SELECT COUNT(*) FROM terminology WHERE pid IS NULL").fetchone()[0]
n_chi = mirB.execute("SELECT COUNT(*) FROM terminology WHERE pid IS NOT NULL").fetchone()[0]
n_ex = mirB.execute("SELECT COUNT(*) FROM data_training").fetchone()[0]
print(f"镜像终态: {n_par} 父组 + {n_chi} 子别名 + {n_ex} 示例（W15-D6: 25/114/29 → 今日 25/114/29 数字守恒，内容换新）")

In [ ]:
# ---- 21 题电池（检索语义复刻：术语=单向子串拉全组；示例=双向子串）----
def term_index(conn):
    idx = []
    for pid, w, d in conn.execute("SELECT id,word,description FROM terminology WHERE pid IS NULL"):
        aliases = [r[0] for r in conn.execute("SELECT word FROM terminology WHERE pid=?", (pid,))]
        idx.append({"word": w, "description": d or "", "aliases": aliases})
    return idx
def hit_terms(q, idx):
    return [g for g in idx if (g["word"] in q) or any(a in q for a in g["aliases"])]
def hit_example(q, exs):
    return [(qq, ss) for qq, ss in exs if (q in qq) or (qq in q)]

idx_A, idx_B = term_index(mirA), term_index(mirB)
exs_A = list(mirA.execute("SELECT question,description FROM data_training"))
exs_B = list(mirB.execute("SELECT question,description FROM data_training"))
battery = [("Q%d" % (i + 1), "Q", e["question"]) for i, e in enumerate(sqls)]
battery += [("V1", "V", "A101 为什么租不出去？"), ("V2", "V", "现在有哪些空着的铺？"),
            ("V3", "V", "星河中心有哪些入驻商户？"), ("V4", "V", "合同 CONT_DEMO_001 下面挂了哪些铺？"),
            ("V5", "V", "铺位 L2-02 现在是什么状态？"),
            ("P1", "P", "本月招商线索转化漏斗怎么样？"), ("P2", "P", "各项目工单完成率如何？"),
            ("P3", "P", "当前项目租赁结构指标怎么样？"),
            ("S1", "S", "昨日全场销售额、客流、坪效分别是多少？"), ("S2", "S", "未来 90 天有哪些合同到期？")]
results = []
for tid, cat, q in battery:
    results.append({"id": tid, "cat": cat,
                    "A_term": bool(hit_terms(q, idx_A)), "B_term": bool(hit_terms(q, idx_B)),
                    "A_ex": bool(hit_example(q, exs_A)), "B_ex": bool(hit_example(q, exs_B))})
covA = (sum(1 for r in results if r["A_term"]), sum(1 for r in results if r["A_ex"]))
covB = (sum(1 for r in results if r["B_term"]), sum(1 for r in results if r["B_ex"]))
print("21 题电池: 术语命中 基线 %d → %d | 示例命中 基线 %d → %d" % (covA[0], covB[0], covA[1], covB[1]))
print("守恒检查（S 组防回归）:", all(r["B_ex"] for r in results if r["cat"] == "S"))
assert covB == (19, 13), "电池数字与 W15-D6 基线漂移"
assert all(r["B_ex"] for r in results if r["cat"] == "S")

# ---- A101 依据链 L1-L3（修正后 description 注入复验）----
Q = "A101 铺位为什么不能出租？"
grp = next(g for g in hit_terms(Q, idx_B) if "A101" in g["aliases"])
assert "1:在租;2:空置" in grp["description"], "修正后的 DDL 枚举口径未随组注入"
sql_a101 = ("SELECT p.STORE_NAME, p.POSITION_CODE, p.POSITION_NAME, p.POSITION_STATE, p.CONT_NO, p.END_DATE, "
            "c.TENANT_NAME, c.CONT_STATE FROM bi_d_position p LEFT JOIN bi_d_contract c ON p.CONT_NO = c.CONT_NO "
            "WHERE p.POSITION_CODE = 'LOC_DEMO_L101'")
cur = db.execute(sql_a101); acols = [d[0] for d in cur.description]; arow = cur.fetchone()
A = dict(zip(acols, arow))
contrast = db.execute("SELECT POSITION_CODE, POSITION_NAME, POSITION_STATE, CONT_NO FROM bi_d_position "
                      "WHERE POSITION_CODE = 'LOC_DEMO_L202'").fetchone()
L1 = arow is not None and A["POSITION_CODE"] == "LOC_DEMO_L101" and A["STORE_NAME"] == "星河购物中心"
L2 = A["CONT_NO"] == "CONT_DEMO_001" and A["TENANT_NAME"] == "云巷咖啡" and contrast[2] == 2
L3 = "1:在租;2:空置" in grp["description"]
print(f"A101 链: L1 身份路径={'PASS' if L1 else 'FAIL'} | L2 业务链={'PASS' if L2 else 'FAIL'}（对照行 {contrast[0]} STATE={contrast[2]} 空置）| L3 规则注入={'PASS' if L3 else 'FAIL'}（DDL 枚举口径随 description 进入 prompt）")
assert L1 and L2 and L3
a101_chain = {"L1": "PASS", "L2": "PASS", "L3": "PASS", "L4": "TODO", "L5": "TODO", "L1_prime": "TODO"}

In [ ]:
# ---- S6 回执换新（含 domain-semantic-pack-contract 对照项）----
contract_alignment = {
 "R1 身份/版本/兼容声明": "ALIGNED——pack_id=lnkcre-mall-ops-demo-pack v0.1.2 + compatibility 声明；demo profile 未注册（先于框架存在），fail-closed 注册路径留给生产 pack（W17+）",
 "R2 术语映射在 Profile 允许清单内": "PARTIAL——demo 对象宇宙 11/11 合法；生产 analysis 白名单 0/11（设计域使然，Identity 登记 9 血缘即重生成输入）",
 "R3 示例问句+语义路由": "PARTIAL——11 条示例=问句模式；结构化 routing entries 未声明（starter 通道按子串检索），W17 契约化路由",
 "R4 能力+结果解释": "PARTIAL——description 注入即解释且只引用 demo 对象；capabilities 声明结构未建",
 "R5 治理函数引用注册键": "N/A-demo——demo 无 governed function；生产侧 gov_ar_aging_summary 已入 Identity 登记（今日①）",
 "R6 不越权/不放宽 Profile": "ALIGNED——pack 纯声明式零权限语义；specific_ds 圈定作用域",
 "R7 跨域语义不隐式共享": "ALIGNED——specific_ds=1 绑定单数据源，不进 oid 共享池（W15-D2 决策②）",
 "R8 声明式 dev-scoped": "ALIGNED——无可执行逻辑入 Core；dev reference 定位显式声明",
}
receipt = {
 "imported_at": datetime.datetime.now().isoformat(timespec="seconds"),
 "executor": "W16-D3 ipynb（生成器 v0.1.2 重出后 S6 验证链重跑：upsert 幂等 + 21 题电池 + e2e + A101 L1-L3）",
 "datasource": {"name": DS_NAME, "id": DS_ID, "binding": "specific_ds=true"},
 "pack": {"pack_id": "lnkcre-mall-ops-demo-pack", "version": "0.1.2",
          "terms": len(terms), "aliases": sum(len(g["other_words"]) for g in terms),
          "sql_examples": len(sqls), "fingerprints": {f: sha16(f"{PACK_DIR}/{f}") for f in ["terminology.json", "sql_examples.json"]}},
 "upsert": {"first_run": list(b1), "second_run": list(b2), "convergence": "PASS（全 0）"},
 "coverage_ab": {"battery": len(results), "term_base": covA[0], "term_after": covB[0],
                 "example_base": covA[1], "example_after": covB[1],
                 "e2e_exec_ok": sum(1 for r in rows_meta if r[4] == 1), "e2e_with_example": 11},
 "a101_chain": a101_chain,
 "value_domain": {"method": "DDL COMMENT 枚举 + 列类型 + 种子值域 三层对账（生成器内建，独立复算一致）",
                  "pass": 9, "na": 2, "fail": 0,
                  "fixes": ["#2 POSITION_STATE='空置'→=2（DDL 枚举 1:在租;2:空置，真跑 0行→1行）",
                            "#8 BILLMONTH=12→BILLYEAR='2024' AND BILLMONTH='12'（varchar 类型对齐，复验新发现）",
                            "铺位/空置组 description 口径同步改 DDL 枚举"]},
 "contract_alignment": contract_alignment,
 "identity_registry": {"file": "semantic-model/governance/identity-analysis-anchors.yaml",
                       "objects": 20, "demo_lineage": 9, "no_lineage": 11, "mapping_gaps": 4,
                       "spec_sha256_16": sha16(WL_SPEC)},
 "p2_check": "trade/industry 字典零新增 canonical（industry_dict_aliases/trade_definitions 8/28 前在册）；canonical 472→477（工程条件库5+门店台账1−mig1）→ W39 digest",
 "whitelist_compliance": {"in_prod_whitelist": "0/11（demo 数据源设计域；血缘桥已登记，W17+ 绑定重生成）",
                          "mallcre_valid": "11/11"},
 "source": {"ontology_sha256_16": sha16("/root/docs/lanlnk/config/ontology/business-ontology.yaml"),
            "semantic_model": "v0.1.1", "generator": "generate_pack.py v0.1.2",
            "lnkcre_head": "0392e107", "starter_pack": "/root/LnkChatBI/backend/scripts/mall_ops_starter_pack"},
}
json.dump(receipt, open(f"{PACK_DIR}/import-receipt.json", "w"), ensure_ascii=False, indent=1)
print("S6 回执换新落盘: import-pack/import-receipt.json")
print("契约对照项（8 条）:")
for k, v in contract_alignment.items():
    print(" ", k, "→", v.split("——")[0])

## §5 S1 周验基线登记（G-05 锚点 + Identity 锚点进 digest 检查单）

D2 明日连接兑现：两套锚点从今日起进 S1 周验基线——W39 digest 增「锚点漂移」检查项（OK/DRIFTED/BROKEN 三级判决入 digest 正文）。

In [ ]:
s1 = f"""# S1 周验基线登记（W16-D3 生效：W39 digest 起增锚点漂移项）
registered: "2026-09-16"
w39_baseline:
  heads: {{lnkcre: 0392e107, docs: c3d08d6, LnkChatBI: c8927267}}
  ontology_sot_fingerprint: bf550bc24de66813   # config/ontology/business-ontology.yaml（W38 复验未漂移）
  canonical_tables: {{count: 477, snapshot_0828: 472,
    delta: "+engineering_condition_items/+engineering_condition_mirror_repairs/+engineering_condition_template_items/+engineering_condition_template_projects/+engineering_condition_templates/+store_change_records -mig_000202_hazard_default_due_days_backup"}}
checks:
  - id: S1-三仓对账
    what: behind/HEAD + wave 主题 + P0-P3 评级（S1 常规）
  - id: S1-指纹复验
    what: ontology SoT sha256_16 复算比对（漂移→G-01 家族裁决）
  - id: S1-G05锚点漂移
    what: python3 semantic-model/governance/ci/frozen_effect_ci.py anchors → 15 锚 OK/DRIFTED/BROKEN 判决入 digest
    new_in: W39（W16-D2 交付的锚点从本基线起进周验）
  - id: S1-Identity锚点复验
    what: governance/identity-analysis-anchors.yaml 20 对象 spec 在册复验（同三级判决语义）
    new_in: W39（W16-D3 交付）
  - id: S1-canonical漂移
    what: canonical_tables.txt 行数 477 基线比对（增量→熵基线刷新 v0.2 议程）
governance: 白名单 spec 或 effect 实现变更 → 锚点 BROKEN → 须带 openspec change（与 ORE-1 同门：变化留痕，不是禁止变化）
"""
open(f"{SEM}/sync/s1-baseline-checklist.yaml", "w").write(s1)
chk = yaml.safe_load(open(f"{SEM}/sync/s1-baseline-checklist.yaml"))
assert len(chk["checks"]) == 5 and chk["w39_baseline"]["heads"]["lnkcre"] == "0392e107"
print("S1 周验基线落盘: sync/s1-baseline-checklist.yaml（5 检查项，其中 2 项为本周新增锚点漂移项）")

## §6 结论：Today's Question 的回答

**为什么修正必须回生成器源头？**

因为 pack 是**投影**（projection），不是**源**。`terminology.json`/`sql_examples.json` 是生成器从 TIER1 源数据 + ontology SoT 推导出的产物；直接改 pack = 在投影上打补丁 = 下次重生成时补丁必然丢失，且**没有任何机器证据能区分「手改的 pack」和「生成的 pack」**。修正回源后：① 修正进 changelog（可审计）；② 回归哨兵进断言（`POSITION_STATE = 2` 必须在、`= '空置'` 必须不在——防脚手架自身回归）；③ 重出全链数字/指纹换新，S6 回执与生成物重新自洽。这正是分层 SoT 宪章在消费面的实例：**改投影是漂移，改源是演进**——两者的区别由 manifest 指纹链机器可证。

**验证汇总**：值域 11/11（9 PASS + 2 N/A，0 FAIL）｜e2e 执行 11/11｜电池 2→19 / 2→13 守恒｜A101 L1-L3 PASS（修正口径随组注入）｜幂等收敛全 0｜Identity 20/20 登记（9 血缘 + 11 无血缘 + 4 缺口显式）｜契约对照 8 项（4 ALIGNED / 3 PARTIAL / 1 N/A，诚实标注）。

**明日（W16-D4）**：G-01 治理头 change 立案窗口跟进（docs 仓受理状态核查 + 对账报告按 W38 digest 发现的 evidence-chain 格式范本深化）；两套锚点（G-05 + Identity）做一次周验预演（跑 anchors 门 + identity 复验，为 W39 digest 排练）；G-04 v0.2 素材整理（V2 口语缺口「空着的铺」补录候选 + 通用词『项目』误触——今日电池再次复现，进 v0.2 别名批次的优先级证据）。